In [1]:
import fitz, os, re, sys, torch
import pandas as pd

sys.path.append(os.path.join(os.getcwd(), 'src'))
import config
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter
from huggingface_hub import login

from dotenv import load_dotenv

load_dotenv()

login(os.environ['HF_TOKEN'])

def get_nlp_tools() -> tuple[RecursiveCharacterTextSplitter, SentenceTransformer]:
    #loading embeddings model
    # Check if CUDA is available
    device = "cuda" if torch.cuda.is_available() else "cpu"

    embeddings = SentenceTransformer("google/embeddinggemma-300m", device=device)
    #Creating text splitter
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=int(config.db_config['chunk_size']),
        chunk_overlap=int(config.db_config['chunk_overlap'])
    )
    return splitter, embeddings

splitter, embeddings = get_nlp_tools()

c:\Users\User\miniconda3\envs\motos\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
Loading weights: 100%|██████████| 314/314 [00:00<00:00, 4468.08it/s]


In [2]:
def get_output_path(base_path: str, file: str) -> str:
    start = base_path.split('\\input\\')[0]
    end = base_path.split('\\input\\')[-1]
    output_path = os.path.join(start, 'output', end)
    return os.path.join(output_path, f"{file.split('.')[0]}.parquet")

def process_pdf(text: list[str], base_path: str, file: str) -> None:
    text = [re.sub('(\n* *\n)+', '\n', page) for page in text]
    text = [re.sub(' +', ' ', page).strip().lower() for page in text]
    text = [[re.sub('\n', ' ', chunk) for chunk in splitter.split_text(page)] for page in text]
    text = [pd.DataFrame({
            'text': page,
            'page': [i+1 for _ in range(len(page))],
            'chunk': range(1, len(page)+1)
        }) for i, page in enumerate(text) if len(page)]
    text = pd.concat(text).reset_index(drop=True).assign(file=file)
    text["embedding"] = [emb.tolist() for emb in embeddings.encode(text.text.tolist())]
    output_path = get_output_path(base_path, file)
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    text.to_parquet(output_path)

def process_database(base_path: str, errors: dict):
    folder = os.path.dirname(base_path)
    file = os.path.basename(base_path)
    output_path = get_output_path(folder, file)
    if os.path.isdir(base_path):
        for element in os.listdir(base_path):
            process_database(os.path.join(base_path, element), errors)
    elif not os.path.exists(output_path):
        try:
            print(f'processing file {file}')
            doc = fitz.open(os.path.join(base_path, file))
            text = [page.get_text() for page in doc]
            process_pdf(folder, file, text)
        except Exception as e:
            errors[base_path] = str(e)

In [3]:
base_path, errors = r'C:\Users\User\Desktop\projects\Chatbot-Taller-Motos\input', dict()
process_database(base_path, errors)

processing file BAJAJ PLATINA 100 MANUAL TALLER 2.pdf
processing file BAJAJ PULSAR 180 MANUAL TALLER.pdf
processing file BAJAJ PULSAR NS200 - AS200 MANUAL USUARIO.pdf
processing file BMW 650 DIAGRAMA ELECTRICO.jpg
processing file BMW F800GS WIRING DIAGRAM 2010.jpg
processing file BMW F800GS WIRING DIAGRAM.JPG
processing file HARLEY DAVIDSON SPORTER (59-85 V-TWINS 86-87) GLIDES &ELECTRA GLIDES (59-84) EVOLUTION V-TWINS (84-86) MANUAL TALLER.pdf
processing file HERO GLAMOUR MANUAL TALLER.pdf
processing file HERO HF DELUXE MANUAL TALLER.pdf
processing file HERO HONDA CD100 MANUAL TALLER.pdf
processing file HERO HUNK 150 MANUAL TALLER.pdf
processing file HERO HUNK160R FI MANUAL TALLER 2022.pdf
processing file HERO KARIZMA ZMR 225 MANUAL TALLER.pdf
processing file HERO PASION PRO MANUAL TALLER.pdf
processing file HONDA C70 MANUAL TALLER 1982.pdf
processing file HONDA CB750F2 MANUAL TALLER 1992.pdf
processing file HONDA CBR250 MANUAL TALLER.pdf
processing file HONDA CBR600 (K00) MANUAL TALLE

In [4]:
import pytesseract
import cv2
import numpy as np

pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

def process_image(image) -> np.array:
    # converting bytes → NumPy array
    image = np.frombuffer(image, np.uint8)

    # encoding the image with OpenCV
    image = cv2.imdecode(image, cv2.IMREAD_COLOR)

    image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    return cv2.adaptiveThreshold(
        image, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY, 31, 2
    )

def get_pdf_images(file: str) -> list:
    doc = fitz.open(file)

    pdf_images = []
    # Iterar por las páginas
    for page_index in range(len(doc)):
        page = doc[page_index]
        # Obtener las imágenes de la página
        images = page.get_images(full=True)
        
        for img in images:
            xref = img[0]  # referencia interna de la imagen
            base_image = doc.extract_image(xref)
            pdf_images.append(process_image(base_image["image"]))
    return pdf_images

In [ ]:
errors_image = dict()
for base_path in errors.keys():
    folder = os.path.dirname(base_path)
    file = os.path.basename(base_path)
    output_path = get_output_path(folder, file)
    if not os.path.exists(output_path):
        try:
            print(file)
            pdf_images = get_pdf_images(base_path)
            text = [pytesseract.image_to_string(img, lang='"spa+eng') for img in pdf_images]
            process_pdf(text, base_path, file)
        except Exception as e:
            errors_image[folder] = str(e)

In [ ]:
import boto3, json, os, secrets, sys

sys.path.append(os.path.join(os.getcwd(), 'src'))
import config

class MotoClient:
    def __init__(self):
        self.bucket_name = config.db_config['s3_bucket']
        self.s3_client = boto3.client(
            's3vectors',
            aws_access_key_id=config.db_config['aws_access_key_id'],
            aws_secret_access_key=config.db_config['aws_secret_access_key'],
            region_name=config.db_config['aws_region']
        )
        self.bedrock_client = boto3.client("bedrock-runtime", region_name=config.db_config['aws_region'])
    
    def embeed_documents(self, documents: list[str]) -> list[list[float]]:
        embeddings=[]
        #Generate vector embeddings for the input texts
        for text in documents:
                body = json.dumps({
                    "inputText": text,
                    "dimensions": config.db_config['embed_truncate'],
                    "normalize": True
                })    
                # Call Bedrock's embedding API
                response = self.bedrock_client.invoke_model(
                    modelId=config.db_config['embeddings_model'],  # Titan embedding model 
                    body=body,
                    contentType="application/json",
                    accept="application/json"
                )   
                # Parse response
                response_body = json.loads(response['body'].read())
                embeddings.append(response_body['embedding'])
        return embeddings
    
    def insert_vectors(self, texts: list[str], metadatas: list[dict]):
        embeddings = self.embeed_documents(texts)
        vectors = [
            {"key": secrets.token_hex(16), "data": {"float32": embedding}, "metadata": metadata}
            for embedding, metadata in zip(embeddings, metadatas)
        ]
        self.s3_client.put_vectors(
            vectorBucketName=self.bucket_name,
            indexName=config.db_config['s3_index'], 
            vectors=vectors
        )
    
    def clean_vectors(self):
        response = self.s3_client.list_vectors(
            vectorBucketName=self.bucket_name,
            indexName=config.db_config['s3_index']
        )
        ids = [v["key"] for v in response["vectors"]]
        if len(ids):
            self.s3_client.delete_vectors(
                vectorBucketName=self.bucket_name,
                indexName=config.db_config['s3_index'],
                keys=ids
            )
            
    def query_db(self, query: str, top_k: int = 5) -> list[dict]:
        request = json.dumps({
            "inputText": query,
            "dimensions": config.db_config['embed_truncate'],
            "normalize": True
        })

        # Invoke the model with the request and the model ID, e.g., Titan Text Embeddings V2. 
        response = self.bedrock_client.invoke_model(modelId="amazon.titan-embed-text-v2:0", body=request)

        # Decode the model's native response body.
        model_response = json.loads(response["body"].read())

        # Extract and print the generated embedding and the input text token count.
        embedding = model_response["embedding"]

        # Performa a similarity query. You can also optionally use a filter in your query
        query = client.s3_client.query_vectors(
            vectorBucketName=config.db_config['s3_bucket'],
            indexName=config.db_config['s3_index'],
            queryVector={"float32":embedding},
            topK=top_k, 
            filter={"page":"scifi"},
            returnDistance=True,
            returnMetadata=True
        )
        return query["vectors"]
    
client = MotoClient()

In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
import fitz, os, sys

sys.path.append(os.path.join(os.getcwd(), 'src'))
import config

from data_processing.pdf_processing import process_pdf

In [7]:

file = 'AKT 115 KOMFORT MANUAL TALLER.pdf'
base_path = os.path.join(config.path['raw_data'], 'AKT')
doc = fitz.open(os.path.join(base_path, file))
text = [page.get_text() for page in doc]
process_pdf(text, base_path, file)

Uploading vectors: 100%|██████████| 3/3 [00:02<00:00,  1.01batch/s]

207 vectors placed in the index ecta-index.


In [6]:
from commons.llm_utils import AWSClient

aws_client = AWSClient()
aws_client.clean_vectors()

In [ ]:
[result['metadata']['chunk'] for result in results]

['Star Wars: A farm boy joins rebels to fight an evil empire in space',
 'Star Wars: A farm boy joins rebels to fight an evil empire in space']

In [46]:
for number in embeddings[0]:
    if not isinstance(number, float):
        print(number)
        print(type(number))

-0.044670317
<class 'numpy.float32'>
0.06769156
<class 'numpy.float32'>
-0.046710305
<class 'numpy.float32'>
0.002501326
<class 'numpy.float32'>
0.081335895
<class 'numpy.float32'>
-0.06774768
<class 'numpy.float32'>
0.008890456
<class 'numpy.float32'>
0.0027079734
<class 'numpy.float32'>
-0.027175203
<class 'numpy.float32'>
-0.105369814
<class 'numpy.float32'>
0.020297287
<class 'numpy.float32'>
0.040648855
<class 'numpy.float32'>
0.052076973
<class 'numpy.float32'>
0.010886095
<class 'numpy.float32'>
-0.07378449
<class 'numpy.float32'>
0.050947346
<class 'numpy.float32'>
0.11922289
<class 'numpy.float32'>
-0.14906202
<class 'numpy.float32'>
-0.02002539
<class 'numpy.float32'>
-0.078656875
<class 'numpy.float32'>
0.08499919
<class 'numpy.float32'>
0.02683582
<class 'numpy.float32'>
0.031312462
<class 'numpy.float32'>
0.050755482
<class 'numpy.float32'>
-0.12856606
<class 'numpy.float32'>
-0.0643286
<class 'numpy.float32'>
0.09222323
<class 'numpy.float32'>
-0.03847305
<class 'numpy.fl

In [2]:
import os, sys

sys.path.append(os.path.join(os.getcwd(), 'src'))
import config
from commons import get_llm
llm = get_llm()

In [3]:
import random

brands = os.listdir(os.path.join(config.path['raw_data']))
models = os.listdir(os.path.join(config.path['raw_data'], random.choice(brands)))
model = random.choice(models)
model

'AKT MAVERICK MANUAL TALLER (BICICLETA ELECTRICA).pdf'

In [ ]:
import json
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

def extract_moto_models(model: str) -> dict:
    moto_models = ChatPromptTemplate.from_messages([
        ("system", config.prompts["moto_models_template"]),
        MessagesPlaceholder(variable_name="input")
    ])

    result = llm.invoke(
            moto_models.invoke({"input": [("human", model)]})
        )
    return json.loads(result.content)

{}

: 